# PointNet Refactor v4

Some refactor, to come up with a good solution and design for model architecture. Ideally similar to `timm`.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter_max, scatter_mean
from typing import Tuple, Optional, Dict

In [2]:
class TNet(nn.Module):
    """T-Net for learning input transforms, adapted for packed format."""
    def __init__(
        self,
        in_channels: int,
        hidden_dim: int = 64,
        out_channels: Optional[int] = None
    ) -> None:
        super().__init__()
        self.out_channels = out_channels or in_channels
        
        self.mlp1 = nn.Sequential(
            nn.Linear(in_channels, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True)
        )
        
        self.mlp2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True)
        )
        
        # Output transformation matrix
        self.output_transform = nn.Linear(hidden_dim, self.out_channels * in_channels)
        
    def forward(
        self,
        x: torch.Tensor,
        batch: torch.Tensor
    ) -> torch.Tensor:
        """
        Args:
            x: Packed point cloud (N, C)
            batch: Batch assignment for each point (N,)
        Returns:
            Transform matrix for each point (N, out_channels, in_channels)
        """
        feat = self.mlp1(x)
        # Global max pooling
        feat = scatter_max(feat, batch, dim=0)[0][batch]
        feat = self.mlp2(feat)
        
        # Generate transformation matrix
        transform = self.output_transform(feat)
        transform = transform.view(-1, self.out_channels, x.size(1))
        
        # Add identity matrix for stable training
        identity = torch.eye(x.size(1), dtype=x.dtype, device=x.device)
        identity = identity.unsqueeze(0).expand(transform.size(0), -1, -1)
        transform = transform + identity
        
        return transform

class PointNetEncoder(nn.Module):
    """PointNet encoder with input and feature transform networks."""
    def __init__(
        self,
        in_channels: int,
        feat_dim: int = 64,
        global_feat: bool = True,
        feature_transform: bool = True
    ) -> None:
        super().__init__()
        self.global_feat = global_feat
        self.feature_transform = feature_transform
        
        # Input transform network
        self.input_transform = TNet(in_channels)
        
        # First MLP
        self.mlp1 = nn.Sequential(
            nn.Linear(in_channels, feat_dim),
            nn.BatchNorm1d(feat_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feat_dim, feat_dim),
            nn.BatchNorm1d(feat_dim),
            nn.ReLU(inplace=True)
        )
        
        # Feature transform network
        self.feature_transform_net = None
        if feature_transform:
            self.feature_transform_net = TNet(feat_dim)
        
        # Second MLP
        self.mlp2 = nn.Sequential(
            nn.Linear(feat_dim, feat_dim * 2),
            nn.BatchNorm1d(feat_dim * 2),
            nn.ReLU(inplace=True),
            nn.Linear(feat_dim * 2, feat_dim * 4),
            nn.BatchNorm1d(feat_dim * 4),
            nn.ReLU(inplace=True)
        )
        
    def forward(
        self,
        x: torch.Tensor,
        batch: torch.Tensor
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Dict[str, torch.Tensor]]:
        """
        Args:
            x: Packed point cloud (N, C)
            batch: Batch assignment for each point (N,)
        Returns:
            Tuple of:
                - Global or local features
                - Transform matrix (if feature_transform=True)
                - Dictionary of intermediate features
        """
        num_points = x.size(0)
        trans = self.input_transform(x, batch)
        x = torch.bmm(x.unsqueeze(1), trans).squeeze(1)
        
        # First MLP
        point_feat = self.mlp1(x)
        
        # Feature Transform
        trans_feat = None
        if self.feature_transform_net is not None:
            trans_feat = self.feature_transform_net(point_feat, batch)
            point_feat = torch.bmm(point_feat.unsqueeze(1), trans_feat).squeeze(1)
        
        # Second MLP
        point_feat = self.mlp2(point_feat)
        
        # Global feature
        global_feat = scatter_max(point_feat, batch, dim=0)[0]
        
        if self.global_feat:
            return global_feat[batch], trans_feat, {'point_feat': point_feat}
        else:
            global_feat_expanded = global_feat[batch]
            return torch.cat([point_feat, global_feat_expanded], dim=1), trans_feat, {
                'point_feat': point_feat,
                'global_feat': global_feat
            }

class PointNet(nn.Module):
    """Complete PointNet model for classification."""
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        feat_dim: int = 64,
        feature_transform: bool = True,
        dropout: float = 0.3
    ) -> None:
        super().__init__()
        self.encoder = PointNetEncoder(
            in_channels=in_channels,
            feat_dim=feat_dim,
            global_feat=True,
            feature_transform=feature_transform
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim * 4, feat_dim * 2),
            nn.BatchNorm1d(feat_dim * 2),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(feat_dim * 2, feat_dim),
            nn.BatchNorm1d(feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(feat_dim, num_classes)
        )
        
    def forward(
        self,
        x: torch.Tensor,
        batch: torch.Tensor
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Dict[str, torch.Tensor]]:
        """
        Args:
            x: Packed point cloud (N, C)
            batch: Batch assignment for each point (N,)
        Returns:
            Tuple of:
                - Classification logits
                - Transform matrix (if feature_transform=True)
                - Dictionary of intermediate features
        """
        global_feat, trans_feat, features = self.encoder(x, batch)
        logits = self.classifier(global_feat)
        
        # Get per-cloud logits
        logits = scatter_mean(logits, batch, dim=0)
        
        return logits, trans_feat, features

def feature_transform_regularizer(trans: torch.Tensor) -> torch.Tensor:
    """Compute regularization loss for the feature transform matrix."""
    d = trans.size()[1]
    identity = torch.eye(d, dtype=trans.dtype, device=trans.device)
    identity = identity.unsqueeze(0).expand_as(trans)
    loss = torch.mean(torch.norm(
        torch.bmm(trans, trans.transpose(2, 1)) - identity, dim=(1, 2)
    ))
    return loss

In [3]:
# Example usage
model = PointNet(
    in_channels=3,  # For XYZ coordinates
    num_classes=40,  # Number of categories
    feat_dim=64,
    feature_transform=True
)

# Forward pass with packed points
points = torch.randn(1000, 3)  # 1000 points, 3 features each
batch = torch.zeros(1000, dtype=torch.long)  # All points in same cloud
logits, trans_feat, features = model(points, batch)

# If using feature transform, add regularization loss
if trans_feat is not None:
    reg_loss = feature_transform_regularizer(trans_feat)

In [5]:
reg_loss

tensor(7.6018, grad_fn=<MeanBackward0>)

### With features

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter_max, scatter_mean
from typing import Tuple, Optional, Dict

class TNet(nn.Module):
    """T-Net for learning input transforms, adapted for packed format."""
    def __init__(
        self,
        in_channels: int,
        hidden_dim: int = 64,
        out_channels: Optional[int] = None
    ) -> None:
        super().__init__()
        self.out_channels = out_channels or in_channels
        
        self.mlp1 = nn.Sequential(
            nn.Linear(in_channels, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True)
        )
        
        self.mlp2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True)
        )
        
        self.output_transform = nn.Linear(hidden_dim, self.out_channels * in_channels)
        
    def forward(
        self,
        x: torch.Tensor,
        batch: torch.Tensor
    ) -> torch.Tensor:
        feat = self.mlp1(x)
        feat = scatter_max(feat, batch, dim=0)[0][batch]
        feat = self.mlp2(feat)
        
        transform = self.output_transform(feat)
        transform = transform.view(-1, self.out_channels, x.size(1))
        
        identity = torch.eye(x.size(1), dtype=x.dtype, device=x.device)
        identity = identity.unsqueeze(0).expand(transform.size(0), -1, -1)
        transform = transform + identity
        
        return transform

class PointNetEncoder(nn.Module):
    """PointNet encoder with support for additional point features."""
    def __init__(
        self,
        coord_channels: int = 3,
        feature_channels: int = 0,
        feat_dim: int = 64,
        global_feat: bool = True,
        feature_transform: bool = True
    ) -> None:
        super().__init__()
        self.global_feat = global_feat
        self.feature_transform = feature_transform
        self.feature_channels = feature_channels
        
        # Transform network for coordinates only
        self.input_transform = TNet(coord_channels)
        
        # First MLP - processes transformed coordinates and features
        self.mlp1 = nn.Sequential(
            nn.Linear(coord_channels + feature_channels, feat_dim),
            nn.BatchNorm1d(feat_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feat_dim, feat_dim),
            nn.BatchNorm1d(feat_dim),
            nn.ReLU(inplace=True)
        )
        
        # Feature transform network
        self.feature_transform_net = None
        if feature_transform:
            self.feature_transform_net = TNet(feat_dim)
        
        # Second MLP
        self.mlp2 = nn.Sequential(
            nn.Linear(feat_dim, feat_dim * 2),
            nn.BatchNorm1d(feat_dim * 2),
            nn.ReLU(inplace=True),
            nn.Linear(feat_dim * 2, feat_dim * 4),
            nn.BatchNorm1d(feat_dim * 4),
            nn.ReLU(inplace=True)
        )
        
    def forward(
        self,
        coords: torch.Tensor,
        features: Optional[torch.Tensor],
        batch: torch.Tensor
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Dict[str, torch.Tensor]]:
        """
        Args:
            coords: Packed point coordinates (N, 3)
            features: Optional packed point features (N, F)
            batch: Batch assignment for each point (N,)
        Returns:
            Tuple of:
                - Global or local features
                - Transform matrix (if feature_transform=True)
                - Dictionary of intermediate features
        """
        # Transform coordinates
        trans = self.input_transform(coords, batch)
        transformed_coords = torch.bmm(coords.unsqueeze(1), trans).squeeze(1)
        
        # Combine transformed coordinates with additional features
        if features is not None:
            x = torch.cat([transformed_coords, features], dim=1)
        else:
            x = transformed_coords
        
        # First MLP
        point_feat = self.mlp1(x)
        
        # Feature Transform
        trans_feat = None
        if self.feature_transform_net is not None:
            trans_feat = self.feature_transform_net(point_feat, batch)
            point_feat = torch.bmm(point_feat.unsqueeze(1), trans_feat).squeeze(1)
        
        # Second MLP
        point_feat = self.mlp2(point_feat)
        
        # Global feature
        global_feat = scatter_max(point_feat, batch, dim=0)[0]
        
        if self.global_feat:
            print(f"{batch.shape = }")
            print(f"{point_feat.shape = }")
            print(f"{global_feat.shape = }")
            print(f"{global_feat[batch].shape = }")
            print(f"{global_feat = }")
            print(f"{global_feat[batch] = }")
            return global_feat[batch], trans_feat, {'point_feat': point_feat}
        else:
            global_feat_expanded = global_feat[batch]
            return torch.cat([point_feat, global_feat_expanded], dim=1), trans_feat, {
                'point_feat': point_feat,
                'global_feat': global_feat
            }

class PointNet(nn.Module):
    """Complete PointNet model with support for additional point features."""
    def __init__(
        self,
        num_classes: int,
        coord_channels: int = 3,
        feature_channels: int = 0,
        feat_dim: int = 64,
        feature_transform: bool = True,
        dropout: float = 0.3
    ) -> None:
        super().__init__()
        self.encoder = PointNetEncoder(
            coord_channels=coord_channels,
            feature_channels=feature_channels,
            feat_dim=feat_dim,
            global_feat=True,
            feature_transform=feature_transform
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim * 4, feat_dim * 2),
            nn.BatchNorm1d(feat_dim * 2),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(feat_dim * 2, feat_dim),
            nn.BatchNorm1d(feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(feat_dim, num_classes)
        )
        
    def forward(
        self,
        coords: torch.Tensor,
        features: Optional[torch.Tensor],
        batch: torch.Tensor
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Dict[str, torch.Tensor]]:
        """
        Args:
            coords: Packed point coordinates (N, 3)
            features: Optional packed point features (N, F)
            batch: Batch assignment for each point (N,)
        Returns:
            Tuple of:
                - Classification logits
                - Transform matrix (if feature_transform=True)
                - Dictionary of intermediate features
        """
        global_feat, trans_feat, features_dict = self.encoder(coords, features, batch)
        logits = self.classifier(global_feat)
        
        # Get per-cloud logits
        logits = scatter_mean(logits, batch, dim=0)
        
        return logits, trans_feat, features_dict

In [25]:
# Initialize model with RGB support
model = PointNet(
    num_classes=40,
    coord_channels=3,    # XYZ coordinates
    feature_channels=3,  # RGB values
    feat_dim=64,
    feature_transform=True,
)

# Example forward pass with coordinates and RGB features
coords = torch.randn(1000, 3)     # XYZ coordinates
rgb = torch.rand(1000, 3)         # RGB values
# Batch of two point clouds
batch = torch.tensor([0] * 600 + [1] * 400, dtype=torch.long)

# Forward pass with both coordinates and features
logits, trans_feat, features = model(coords, rgb, batch)

batch.shape = torch.Size([1000])
point_feat.shape = torch.Size([1000, 256])
global_feat.shape = torch.Size([2, 256])
global_feat[batch].shape = torch.Size([1000, 256])
global_feat = tensor([[1.4115, 3.6119, 2.3040, 3.6977, 2.2472, 1.1783, 2.0717, 2.1203, 1.4503,
         2.1163, 2.8566, 3.6913, 0.8229, 2.1073, 1.7832, 3.5096, 2.2288, 5.3537,
         2.1997, 2.6438, 1.4923, 2.6734, 2.1852, 2.2660, 1.0978, 2.6161, 3.2722,
         2.2145, 0.6108, 3.8506, 2.7336, 2.7345, 1.8700, 0.8395, 2.9048, 2.7269,
         2.2111, 1.5218, 2.0050, 1.6164, 2.2784, 1.1438, 2.5548, 1.4479, 2.8805,
         1.9281, 2.5477, 3.9309, 1.8224, 1.4331, 2.0227, 3.3051, 3.1772, 2.9289,
         2.5029, 2.2034, 2.5508, 3.0409, 2.1829, 1.1688, 2.7525, 1.0331, 2.8189,
         2.2195, 3.7144, 2.3149, 1.8012, 1.4279, 3.3845, 2.6072, 1.8226, 2.6615,
         3.4053, 1.7837, 2.0603, 2.2251, 2.2073, 1.5769, 2.4256, 3.0551, 3.2042,
         3.9558, 1.7291, 2.8036, 2.7497, 3.4935, 3.8562, 1.7009, 1.6398, 1.7042,
        

### Segmentation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter_max, scatter_mean
from typing import Tuple, Optional, Dict

# Note: TNet and base PointNetEncoder classes remain the same as previous implementation

class SegmentationEncoder(PointNetEncoder):
    """PointNet encoder specifically adapted for segmentation tasks."""
    def __init__(
        self,
        coord_channels: int = 3,
        feature_channels: int = 0,
        feat_dim: int = 64,
        feature_transform: bool = True
    ) -> None:
        super().__init__(
            coord_channels=coord_channels,
            feature_channels=feature_channels,
            feat_dim=feat_dim,
            global_feat=False,  # Important: we need local features for segmentation
            feature_transform=feature_transform
        )

class SegmentationDecoder(nn.Module):
    """Decoder network for point-wise segmentation."""
    def __init__(
        self,
        feat_dim: int = 64,
        num_classes: int = 50,
        dropout: float = 0.3
    ) -> None:
        super().__init__()
        
        # Input size is feat_dim * 4 (from encoder) + feat_dim * 4 (global feature)
        input_dim = feat_dim * 8
        
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, feat_dim * 2),
            nn.BatchNorm1d(feat_dim * 2),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            
            nn.Linear(feat_dim * 2, feat_dim),
            nn.BatchNorm1d(feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            
            nn.Linear(feat_dim, feat_dim // 2),
            nn.BatchNorm1d(feat_dim // 2),
            nn.ReLU(inplace=True),
            
            nn.Linear(feat_dim // 2, num_classes)
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Concatenated local and global features (N, feat_dim * 8)
        Returns:
            Point-wise segmentation logits (N, num_classes)
        """
        return self.mlp(x)

class PointNetSegmentation(nn.Module):
    """PointNet model for semantic segmentation tasks."""
    def __init__(
        self,
        num_classes: int,
        coord_channels: int = 3,
        feature_channels: int = 0,
        feat_dim: int = 64,
        feature_transform: bool = True,
        dropout: float = 0.3
    ) -> None:
        super().__init__()
        
        self.encoder = SegmentationEncoder(
            coord_channels=coord_channels,
            feature_channels=feature_channels,
            feat_dim=feat_dim,
            feature_transform=feature_transform
        )
        
        self.decoder = SegmentationDecoder(
            feat_dim=feat_dim,
            num_classes=num_classes,
            dropout=dropout
        )
    
    def forward(
        self,
        coords: torch.Tensor,
        features: Optional[torch.Tensor],
        batch: torch.Tensor
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Dict[str, torch.Tensor]]:
        """
        Args:
            coords: Packed point coordinates (N, 3)
            features: Optional packed point features (N, F)
            batch: Batch assignment for each point (N,)
        Returns:
            Tuple of:
                - Per-point segmentation logits (N, num_classes)
                - Transform matrix (if feature_transform=True)
                - Dictionary of intermediate features
        """
        # Get concatenated local and global features
        x, trans_feat, feat_dict = self.encoder(coords, features, batch)
        
        # Predict per-point segmentation
        logits = self.decoder(x)
        
        return logits, trans_feat, feat_dict

# Training-specific components
class PointNetSegmentationLoss(nn.Module):
    """Combined segmentation and feature transform regularization loss."""
    def __init__(self, mat_diff_loss_scale: float = 0.001) -> None:
        super().__init__()
        self.mat_diff_loss_scale = mat_diff_loss_scale
        
    def forward(
        self,
        pred_logits: torch.Tensor,
        target: torch.Tensor,
        trans_feat: Optional[torch.Tensor]
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        Args:
            pred_logits: Predicted segmentation logits (N, num_classes)
            target: Ground truth labels (N,)
            trans_feat: Optional feature transform matrix
        Returns:
            Tuple of:
                - Total loss
                - Dictionary of individual losses
        """
        seg_loss = F.cross_entropy(pred_logits, target)
        
        # Feature transform regularization
        mat_diff_loss = torch.tensor(0., device=pred_logits.device)
        if trans_feat is not None:
            mat_diff_loss = feature_transform_regularizer(trans_feat)
        
        total_loss = seg_loss + mat_diff_loss * self.mat_diff_loss_scale
        
        return total_loss, {
            'seg_loss': seg_loss,
            'mat_diff_loss': mat_diff_loss
        }